# $xz$-plane optimization

In this notebook we explore the optimization of the entanglement among the set of valid Bloch lengths in two-qutrit systems.

It goes through the generation of figures 2 (d) and 2 (e) of the paper.

# Importations

In [ ]:
# Numerical and scientific python programming
import numpy as np

import matplotlib.pyplot as plt

# Auxiliary python functions
import time
from typing import Any

# Local importations
from moments.bloch import (generate_gell_mann_basis, compute_tensor_basis, compute_subset_index_map,
                           compute_bipartite_region_upper, compute_bipartite_region_ent, compute_bipartite_region_lower)
from moments.optimization import OptimizationResult, opt_moment_preserving_ent

# Saving
from moments.saving import find_project_root

# Hilbert space definition

First we define some variables that will be used through out most of the functions.

For an $N$ qubit system with indices taking values $n = 1, \ldots, N$, we now change the convention to follow python's indexing $n \mapsto n - 1$, such that $N = 0, \ldots, N - 1$.

- The dimenstion of each local Hilbert space is indicated by the variable `dim`: each entry `dim[n]` corresponds to $d_n$.

Variables `pauli_basis`, `local_bases` and `local_basis_sizes` are intermidiate steps to compute `tensor_basis` and `subset_index_map`.

- The variabe `tensor_basis` is a numpy array containing the basis $\{ \mu_i \}_{i = 0}^{d^2 - 1}$ of the Hilbert space $\mathbb H$. Since $\mathcal H = \bigotimes_{n \in \mathbf N} \mathcal H_{n-1}$, then erach basis element is expanded as $\mu_{i_1}^1 \otimes \cdots \otimes \mu_{i_N}^N$. These are ordered in lexycographic order.

- The variable `subset_index_map` is a dictionary that as keys has every possible subset $\mathbf M \subseteq \mathbf N$ of the set of sub-systems. The value of each key corresponds to the indices $i$ of the bloch vector $r_i$ that describe the subsystem $\mathbf M$, according to equation (3) of the paper. This serves to indicate several routines which elements to use if only some subsystems are to be taken into account.

In [ ]:
# Define system parameters.
dn = 3
dim = [dn, dn]
N = len(dim)

# Initialize the Pauli basis of a one-qubit system.
basis = generate_gell_mann_basis(dn)
local_bases = [basis.copy()] * N
local_basis_sizes = [len(lb) for lb in local_bases]

# Compute tensor basis of the N qubit system.
tensor_basis = compute_tensor_basis(local_bases)
# Compute index mappings from basis elements to Bloch vector elements.
subset_index_map = compute_subset_index_map(local_basis_sizes)

# Space parametrization

The space of quantum states in terms of the Bloch lengts is a subset of $\mathbb R^3$, where each coordinate corresponds to one Bloch length and they satisfy several conditions. For more information we refer to Phys. Rev. A 109, 012423 (2024) or section 3.3 of the paper.

Denote $|\vec r_1| := x$, $|\vec r_2| := y$ and $|\vec r_{12}| := z$. The particular conditions for a two-qubit system read:

- $(x, y, z) \in [0, \sqrt2] \times [0, \sqrt2] \times [0, 2\sqrt2]$,
- $z \ge \sqrt2 (x + y) - 2$,
- $z^2 \le 8 + 2 (x^2 + y^2) - 6xy - 3 \sqrt6 |x - y|$.

Moreover, we will be focussing in the the intersection of this region in the first quadrant and the plane $y=0$. This simplifies the conditions to:

- $(x, z) \in [0, \sqrt2] \times [0, 2\sqrt2]$,
- $z \ge \sqrt2 x - 2$,
- $z^2 \le 8 - 3 \sqrt6 x + 2 x^2$.

Besides the boundaries that indicate when there exists a valid quantum states with given Bloch lengths, it is known that at least two different regions exist with different behavours. (i) a region where both separable and entangled states exists with the same Bloch lengths, and (ii) a region were only entangled states lie. These are generated an inequalities, which for two qutrits and the plane $y=0$ read:

- $z \ge \sqrt{2 - x^2}$.

The rest of the region is defined numerically. We import the date from the numerical polytope optimization to define it.

In [ ]:
# Define paths for relevant directories.
PROJECT_ROOT = find_project_root()
data_dir = PROJECT_ROOT / "data" / "examples" / "two_qutrits"

# Import distances to valid quantum states with given Bloch lengths
bloch_distance = np.load(data_dir / "bloch_distance.npz")
xz_distance = bloch_distance["xz"]
Dx, Dz = xz_distance.shape

# Define the domain where the function will be evaluated.
x = np.linspace(0, np.sqrt(2), Dx)
z = np.linspace(0, 2*np.sqrt(2), Dz)

# Construct the two-dimensional coordinate mesh.
X, Z = np.meshgrid(x, z, indexing='ij')

# Define a valid point if the distance to the target moments is below a certain threshold.
indices = np.ndindex(X.shape)
success = np.full_like(X, np.False_)
for idx, jdx in indices:
    if xz_distance[idx, jdx] <= 1e-3:
        success[idx, jdx] = np.True_

In [ ]:
# Plot the regions indicating the separation between the different behavours.
fig, ax = plt.subplots(figsize=(4.5, 8), constrained_layout=True)

mesh = ax.pcolormesh(X, Z, success, shading='auto', vmin=0, vmax=1)

ax.set(xlabel=r"$|| r_1 ||$", ylabel=r"$||r_{1, 2}||$",
       xlim=(0, 1.5), ylim=(0, 3), title="Optimization success")

x = np.linspace(0, np.sqrt(2), 1000)
z_upper = compute_bipartite_region_upper(dn, x, 0)
z_ent = compute_bipartite_region_ent(dn, x, 0)
z_lower = compute_bipartite_region_lower([dn, dn], x, 0)

ax.plot(x, z_upper, 'red', linewidth=2, label=r"$z = \sqrt{8 - 3\sqrt{6}x + 2x^2}$")
ax.plot(x, z_ent, 'orange', linewidth=2, label=r"$z = \sqrt{8 - x^2}$")
ax.plot(x, z_lower, 'blue', linewidth=2, label=r"$z = \max(0, \sqrt{2} x - 2)$")

ax.legend(loc='upper right')
ax.grid(True, alpha=0.5, linestyle="--")
ax.legend()
plt.show()

# Moment space discretization

To iterate over valid quantum states and perform the optimization, we create a grid parametrized by the phisically allowed Bloch lengths.

- The number of points used for the grid coordinates (`Dx` and `Dz`) is read from the polytope characterixtics.

- The variables `x`, and `z` are numpy arrays that discretize each coordinate. Afther that the two-dimensional coordinate mesh is defined with variables `X` and `Z`.

- The varibale `indices` extracts the indices of the valid points in the polytope to avoid iterating over unnecesary points.

In [ ]:
# Determine the number of grid points along each coordinate.
Dx, Dz = success.shape

# Discretize each coordinate.
x = np.linspace(0, np.sqrt(2), Dx)
z = np.linspace(0, 2*np.sqrt(2), Dz)

# Construct the two-dimensional coordinate mesh.
X, Z = np.meshgrid(x, z, indexing='ij')

# Extract the indices of all physically allowed grid points for the loop.
indices = np.where(success)
total_points = len(indices[0])
print(f"Number of points:", total_points)

# Optimization

Here we run the main optimization loop, over the points defined previously on the grid.

- Results of the optimiztion are stored in the numpy array `ent_max`/`ent_min`. We are also storing the time taken for each point to optimize in `times`.

- `run_kwargs` is a keyword arguments passed to the optimization routine. It contains every required argument of the optimization routine.
    - "dim": list of integers. List of local dimensions of the quantum system.
    - "tensor_basis": numpy array. Tensor-product operator basis, shape $(d^2 - 1, d_n, d_n)$.
    - "subset_index_map": dictionary. Keys are tupples of integers (subsets $\mathbf M \subseteq \mathbf N$ of the set of sub-systems). Values are numpy arrays with the indices $i$ of the bloch vector $r_i$ that describe the subsystem $\mathbf M$.
    - "Rt": dictionary. Keys are tupples of integers (subsets $\mathbf M \subseteq \mathbf N$ of the set of sub-systems). Values are the target Bloch vector norms for each subsystem subset.
    - "metric": str. The entanglement metric to optimize. Valid entries are "concurrence", "entanglement_of_formation", "partial_trace_norm" and "negativity". "concurrence" and "entanglement_of_formation" are only implemented for two-qubit systems. The default is "negativity".
    - "optimization": str. Whether to 'minimize' or 'maximize' the metric. The default is "minimize".
    - "cholesky_opt": bool. If True, use Cholesky parametrization for optimization. The default is False.
    - "exact_jac": bool. If True, compute the exact Jacobian for the trace norm metri. the  default is False.
    - "purity_tol": float. Tolerance for purity checks. The default is $10^{-10}$.
    - "psd_tol": float. Tolerance for positive semidefinite checks. The default is $10^{-10}$.
    - "jac_tol": float. Tolerance for Jacobian calculations. The default is $10^{-10}$.
    - "local_maxiter": int. Maximum iterations for the local optimizer. The default is $500$.

- `run_repeat` is a function which optimizes a fixed ammount of times given by `attempts` always keeps the best result. This is specially useful in regions where convergence is difficult to optain in a single pass.

Optimization is only implemented for bipartite systems.

For the maximization problem, we are following a different strategy in each region. In particular, the only entangled and only separable regions only require one round of optimizaiton to output a feasible value. On the other hand, the mixed region is more difficult. Close to the separable region maximum entanglement is very small, making the optimization function almost flat. Most of the times, the algorithm outputs a separable state as maximum entangled when there should exist an actual entangled state. Therefore, in this points we may be interested in runing the algorithm more times, looking for initial points that simplify the optimization.

For the minimization, this problem is not important and a single pass is required for all region.

In [ ]:
def run_repeat(run_kwargs: dict[str, Any], attempts: int = 5) -> OptimizationResult:
    """
    Run an optimization repeatedly and retain the best result.

    The optimization direction is inferred from the ``optimization`` entry in ``run_kwargs``.
    For maximization, the result with the largest final metric is retained.
    For minimization, the result with the smallest final metric is retained.

    Parameters
    ----------
    run_kwargs : dict[str, Any]
        Keyword arguments passed to :func:`opt_moment_preserving_ent`.
    attempts : int, default=5
        Number of independent optimization attempts to perform.

    Returns
    -------
    OptimizationResult
        The optimization result with the best final metric among all attempts.

    Raises
    ------
    ValueError
        If ``attempts`` is less than one.
    """
    # Validate that at least one optimization attempt will be performed.
    if attempts < 1:
        raise ValueError("attempts must be at least 1")
    
    # Determine whether the optimization should maximize or minimize the metric.
    maximize = (run_kwargs["optimization"] == "maximize")

    # Initialize the best result and metric.
    best_result = None
    best_metric = -np.inf if maximize else np.inf

    # Repeat the optimization the requested number of times.
    for _ in range(attempts):
        # Execute one optimization run.
        result = opt_moment_preserving_ent(**run_kwargs)

        # Replace the stored result when this run improves the metric.
        if (maximize and result.metric_final > best_metric) or \
           (not maximize and result.metric_final < best_metric):
            best_metric = result.metric_final
            best_result = result

    assert best_result is not None
    return best_result

In [ ]:
# List to store the time for each point.
times = []
# Store the results of the optimization.
ent_max = np.full_like(X, np.nan, dtype=float)

# Define arguments for the optimization routine.
run_kwargs = {"dim": dim, "tensor_basis": tensor_basis, "subset_index_map": subset_index_map,
              "optimization": "maximize", "metric": "partial_trace_norm", "cholesky_opt": True, "exact_jac": True}

# Iterate over every physically allowed grid point.
for counter, (idx, jdx) in enumerate(zip(indices[0], indices[1]), 1):
    t0 = time.time()

    # Read Bloch lengths for each point.
    x_val = X[idx, jdx]
    z_val = Z[idx, jdx]

    # Construct Boch length constraints.
    Rt = {(1,): float(x_val), (2,): 0.0, (1, 2): float(z_val)}
    run_kwargs["Rt"] = Rt
    
    # Solve the maximization problem with a different strategy in each region.
    optimization_result = opt_moment_preserving_ent(**run_kwargs)
    ent_max[idx, jdx] = optimization_result.metric_final

    tf = time.time()
    times.append(tf-t0)
    
    # Display status of the loop.
    percent_complete = (counter / total_points) * 100
    print(f"\rProgress: {percent_complete:.2f}% ({counter}/{total_points}) | Time for this point: {tf-t0:.3f} s", end="", flush=True)

print(f"\n\nDone!")
print(f"Total time: {sum(times)/60:.2f} min | Average time per point: {sum(times)/total_points:.3f} s", )

In [ ]:
# List to store the time for each point.
times = []
# Store the results of the optimization.
ent_min = np.full_like(X, np.nan, dtype=float)

# Define arguments for the optimization routine.
run_kwargs = {"dim": dim, "tensor_basis": tensor_basis, "subset_index_map": subset_index_map,
              "optimization": "minimize", "metric": "partial_trace_norm", "cholesky_opt": True, "exact_jac": True}

# Iterate over every physically allowed grid point.
for counter, (idx, jdx) in enumerate(zip(indices[0], indices[1]), 1):
    t0 = time.time()

    # Read Bloch lengths for each point.
    x_val = X[idx, jdx]
    z_val = Z[idx, jdx]

    # Construct Boch length constraints.
    Rt = {(1,): float(x_val), (2,): 0.0, (1, 2): float(z_val)}
    run_kwargs["Rt"] = Rt

    # Solve the minimization problem.
    optimization_result = opt_moment_preserving_ent(**run_kwargs)
    ent_min[idx, jdx] = optimization_result.metric_final

    tf = time.time()
    times.append(tf-t0)
    
    # Display status of the loop.
    percent_complete = (counter / total_points) * 100
    print(f"\rProgress: {percent_complete:.2f}% ({counter}/{total_points}) | Time for this point: {tf-t0:.3f} s", end="", flush=True)

print(f"\n\nDone!")
print(f"Total time: {sum(times)/60:.2f} min | Average time per point: {sum(times)/total_points:.3f} s", )

# Save

In [ ]:
# Define paths for relevant directories.
PROJECT_ROOT = find_project_root()
data_dir = PROJECT_ROOT / "data" / "examples" / "two_qutrits"
# Create directories if they don't exist.
data_dir.mkdir(parents=True, exist_ok=True)

np.savez(data_dir / "xz_ent_exp.npz",
         max=ent_max,
         min=ent_min)